In [8]:
import os
import sys

ROOT = os.path.abspath("..")  
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### 1. Why robustness to translation does not automatically imply robustness to rotation?

The output of the following codeblock shows that our alignment performs better for translation but does not do well for rotation because a small rotation has already caused a huge error. The reason is that our alignment code handles positions and scaling and does nothing for rotation. 

In [9]:
from kaktovikAlignmentSimple import simpleAlignment
from HOGFeature import calculateHOG
from DistanceMeasure import euclideanDistance
import cv2
import numpy as np

original = cv2.imread("../img/kaktovik-012_rot_0.jpg")

print("=== Translation (alignment should compensate) ===")
for shift in [10, 30, 50]:
    h, w = original.shape[:2]
    translated = np.full_like(original, 255)
    translated[shift:h, shift:w] = original[0 : h - shift, 0 : w - shift]
    d = euclideanDistance(
        calculateHOG(simpleAlignment(original)),
        calculateHOG(simpleAlignment(translated)),
    )
    print(f"  shift {shift}px  → HOG distance: {d:.4f}")

print()
print("=== Rotation (alignment cannot compensate) ===")
for angle in [10, 30, 45]:
    h, w = original.shape[:2]
    M = cv2.getRotationMatrix2D((w // 2, h // 2), angle, 1.0)
    rotated = cv2.warpAffine(original, M, (w, h), borderValue=(255, 255, 255))
    d = euclideanDistance(
        calculateHOG(simpleAlignment(original)), calculateHOG(simpleAlignment(rotated))
    )
    print(f"  rotation {angle}°  → HOG distance: {d:.4f}")

=== Translation (alignment should compensate) ===
  shift 10px  → HOG distance: 3.0266
  shift 30px  → HOG distance: 12.1287
  shift 50px  → HOG distance: 12.4970

=== Rotation (alignment cannot compensate) ===
  rotation 10°  → HOG distance: 12.0131
  rotation 30°  → HOG distance: 14.9880
  rotation 45°  → HOG distance: 15.4080


### 2. How the computed features can already be used for a simple retrieval task before any classifier is trained?

Without any classification we are just loading all the images and calculating the HOG then calculating the distance of the feature vector of the image with any other image in the dataset and we sort them in ascending order. In other words, the feature descriptions with the smallest euclidean with our image come on top. Then we calculate the top-1, top-3 and top-5 scores which means for example in top-3 if there is at least one image with the same class as our image in the top 3 elements of our sorted HOG list, then we call it a hit and a good allocation to the class. 

In [10]:
GALLERY_DIR = "../img/retrieval"


def get_class(filename):
    return filename.split("_")[0]


def get_hog(path):
    img = cv2.imread(path)
    return calculateHOG(simpleAlignment(img))


image_files = sorted([f for f in os.listdir(GALLERY_DIR) if f.endswith(".jpg")])
image_paths = {f: os.path.join(GALLERY_DIR, f) for f in image_files}

print("Computing HOG descriptors...")
hogs = {f: get_hog(p) for f, p in image_paths.items()}
print(f"Done. {len(hogs)} images loaded.\n")

top1 = top3 = top5 = 0
total = len(image_files)

for query_file in image_files:
    query_class = get_class(query_file)
    query_hog = hogs[query_file]

    distances = []
    for gallery_file, gallery_hog in hogs.items():
        if gallery_file == query_file:
            continue
        d = euclideanDistance(query_hog, gallery_hog)
        distances.append((gallery_file, get_class(gallery_file), d))

    distances.sort(key=lambda x: x[2])

    if distances[0][1] == query_class:
        top1 += 1
    if any(c == query_class for _, c, _ in distances[:3]):
        top3 += 1
    if any(c == query_class for _, c, _ in distances[:5]):
        top5 += 1

print("=== HOG Retrieval Accuracy ===")
print(f"Top-1: {top1}/{total} = {100*top1/total:.1f}%")
print(f"Top-3: {top3}/{total} = {100*top3/total:.1f}%")
print(f"Top-5: {top5}/{total} = {100*top5/total:.1f}%")

Computing HOG descriptors...
Done. 220 images loaded.

=== HOG Retrieval Accuracy ===
Top-1: 163/220 = 74.1%
Top-3: 190/220 = 86.4%
Top-5: 199/220 = 90.5%


### 3. Why MSE is not used for pixel-wise comparison and euclidean is used for feature-descriptor-based comparison?

MSE grows explosively — from 424 to 6323, a 15x increase. It's extremely sensitive to translation even after alignment.
HOG grows much more slowly — from 3.4 to 12.1, roughly 3.5x. And notice it almost plateaus between 30px and 50px — alignment is absorbing most of the shift and HOG stays relatively stable.

Since in classification we don't want translation to affect how we classify a symbol, this robustness is valuable. 

In [14]:
from DistanceMeasure import mseDistance

original = cv2.imread(f"{GALLERY_DIR}/class7_kaktovik-000_rot_0.jpg")

for shift in [10, 30, 50]:
    h, w = original.shape[:2]
    translated = np.full_like(original, 255)
    translated[shift:h, shift:w] = original[0 : h - shift, 0 : w - shift]

    aligned_orig = simpleAlignment(original)
    aligned_trans = simpleAlignment(translated)

    mse = mseDistance(aligned_orig, aligned_trans)
    euc = euclideanDistance(calculateHOG(aligned_orig), calculateHOG(aligned_trans))

    print(f"shift {shift}px → MSE: {mse:.2f}   HOG: {euc:.4f}")

shift 10px → MSE: 424.76   HOG: 3.4255
shift 30px → MSE: 3258.40   HOG: 11.1272
shift 50px → MSE: 6323.44   HOG: 12.1169
